# 02. Climate Obstruction Narratives

This is the longer exercise using your dataset.

Learning outcomes:
1. Build directed and undirected networks from edge lists
2. Understand actor-domain adjacency matrices
3. Use matrix multiplication intuition on subgraphs
4. Compute and interpret centrality measures

## 0) Files Needed

You only need the main data file **`df_climate_obstruction.csv`** — one row per actor–domain link (with columns used in section 2). The notebook builds all edge lists, graphs, and Gephi files from this CSV. The check below verifies the file is present.

In [1]:
# Check that the required data file is present (everything else is built in the notebook)
from pathlib import Path

required = 'df_climate_obstruction.csv'
print(required, 'OK' if Path(required).exists() else 'MISSING')

df_climate_obstruction.csv MISSING


## Chapter 1 Framing for This Dataset

We model a social-media information environment as a network:
- **Actors:** `actor_name`
- **Ties:** links from actors to `domain_clean`
- **Attributes:** `platform`, `account_type` (e.g. twitter_account, facebook_group), `account_anno` (e.g. opinionleader, group), `actor_anno` (e.g. organization, citizen, political party), `emne_profil` (e.g. climate or fringe-right oriented)

This setup lets us ask dyad-, node-, and network-level questions (see below).

## Dyad / Node / Network Questions

At each level of analysis we ask different questions. The table below maps these levels to variables you can compute in this dataset.

- **Dyad level:** Does a specific actor link to a specific domain? (One edge = one dyad.)
- **Node level:** Which actors are most central in the network? (We use the *projected* actor network: two actors are tied if they share at least one domain.)
- **Network level:** How fragmented is the actor network? (E.g. number of components or communities.)

**Question(s):** After running the next cell, in one sentence each: what does the *dyad* row tell you? the *node* row? the *network* row?

In [2]:
# Map each level of analysis to a variable we can compute and to columns in this dataset
import pandas as pd

analysis_map = pd.DataFrame({
    'level': ['dyad', 'node', 'network'],
    'example_variable': ['edge_exists(actor, domain)', 'degree_centrality(actor)', 'number_of_components(graph)'],
    'from_this_dataset': ['actor_name + domain_clean', 'projected actor network', 'whole projected graph']
})
analysis_map

,level,example_variable,from_this_dataset
0,dyad,"edge_exists(actor, domain)",actor_name + domain_clean
1,node,degree_centrality(actor),projected actor network
2,network,number_of_components(graph),whole projected graph


## State vs Event Interpretation

**Events:** If each link is a repeated event (e.g. “actor shared this domain 5 times”), we keep **weight** = count. That lets us see who shares which domains most often.

**State:** If we only care whether a link exists (“actor has ever shared this domain”), we use **binary** edges (0/1). Many measures (e.g. degree) can be computed either way; in this notebook you can switch between weighted and unweighted.

## 1) Setup

Import the libraries we need and set a default figure size for plots. Run this cell first.

In [3]:
# 1) Imports
import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
from itertools import combinations
from networkx.algorithms.community import greedy_modularity_communities

# 2) Default plot size
plt.rcParams['figure.figsize'] = (10, 7)

## 2) Load and Clean Core Variables

We need a table where each row is a link between an **actor** (e.g. a Twitter account) and a **domain** (e.g. a news site). From the full dataset we keep only the columns used in this exercise:

- **For the network:** `actor_name` and `domain_clean` — these define the edges (who links to which domain).
- **For labels later:** `account_anno`, `actor_anno`, `emne_profil`, `platform`, `account_type` — we use these to colour or group nodes in the graph.

**Cleaning** means: strip spaces from text, put domains in lowercase so "NYT.com" and "nyt.com" count as the same, and remove any row where actor or domain is missing or empty. The result is a clean list of actor–domain pairs.

In [5]:
# 1) Load the data and keep only the columns we need
from google.colab import files
uploaded = files.upload()
df = pd.read_csv('df_climate_obstruction.csv')
cols = ['actor_name', 'domain_clean', 'account_anno', 'actor_anno', 'emne_profil', 'platform', 'account_type']
work = df[cols].copy()

# 2) Clean text: strip spaces, lowercase domains so matches are consistent
work['actor_name'] = work['actor_name'].astype(str).str.strip()
work['domain_clean'] = work['domain_clean'].astype(str).str.strip().str.lower()
for c in ['account_anno', 'actor_anno', 'emne_profil', 'platform', 'account_type']:
    work[c] = work[c].astype(str).str.strip()

# 3) Keep only rows that have both an actor and a domain
work = work.dropna(subset=['actor_name', 'domain_clean'])
work = work[(work['actor_name'] != '') & (work['domain_clean'] != '')]

print('Rows after cleaning:', len(work))
work.head()

Saving df_climate_obstruction.csv to df_climate_obstruction.csv
Rows after cleaning: 13346


,actor_name,domain_clean,account_anno,actor_anno,emne_profil,platform,account_type
0,Klimarealisme.dk,nan,opinionleader,organization,climate,facebook,facebook_group
1,Klimarealisme.dk,nettavisen.no,opinionleader,organization,climate,facebook,facebook_group
2,Klimarealisme.dk,indblik.dk,opinionleader,organization,climate,facebook,facebook_group
3,Klimarealisme.dk,nan,opinionleader,organization,climate,facebook,facebook_group
4,Klimarealisme.dk,dr.dk,opinionleader,organization,climate,facebook,facebook_group


## 3) Directed Edge List (Actor → Domain)

**Goal:** One row per (actor, domain) pair, with a **weight** = how many times that actor linked to that domain. We group the cleaned table by actor and domain, then count rows. Columns are renamed to `source` (actor), `target` (domain), and `weight` so networkx can read them later.

In [6]:
# 1) Group by (actor, domain) and count → one row per pair with weight
edge_dir = (
    work.groupby(['actor_name', 'domain_clean'], as_index=False)
    .size()
    .rename(columns={'actor_name': 'source', 'domain_clean': 'target', 'size': 'weight'})
)

print('Directed edges:', len(edge_dir))
edge_dir.head()

Directed edges: 3213


,source,target,weight
0,#Sverigeupproret#,fyens.dk,1
1,"1,8 Miljoner Missnöjda",berlingske.dk,1
2,112Nyt,nan,1
3,180Grader,180grader.dk,112
4,24NYT,24nyt.dk,191


In [7]:
# Save the edge list so we can reuse it or open it in another tool
edge_dir.to_csv('domain_edge_list.csv', index=False)
print('Saved: domain_edge_list.csv')

Saved: domain_edge_list.csv


## 4) Directed Graph and Basic Counts

**Goal:** Build a directed graph where nodes = actors and domains, and each edge is an actor → domain link (with optional weight). We use the edge list from section 3. The result lets us count how many nodes and edges we have.

In [8]:
# Build directed graph from the edge list (nodes = actors + domains)
Gd = nx.from_pandas_edgelist(edge_dir, 'source', 'target', edge_attr='weight', create_using=nx.DiGraph())

print('Directed nodes:', Gd.number_of_nodes())
print('Directed edges:', Gd.number_of_edges())

Directed nodes: 1321
Directed edges: 3213


## 5) Adjacency Matrix (Actors × Domains)

**Goal:** A rectangular table: rows = actors, columns = domains. Entry (i, j) = how many links actor i has to domain j (or 0/1 if we use binary). This is the **incidence** (or bipartite adjacency) form: two node types, so the matrix is not square. We build it with a cross-tabulation of the edge list.

In [9]:
# Cross-tabulate: rows = source (actor), columns = target (domain), values = count
B = pd.crosstab(edge_dir['source'], edge_dir['target'])

print('Shape (actors x domains):', B.shape)
B.iloc[:10, :10]

Shape (actors x domains): (1077, 245)


target,180grader.dk,20min.ch,24nyt.dk,a.msn.com,aalborg.enhedslisten.dk,achgut.com,aftenbladet.no,aftenposten.no,aftonbladet.se,altinget.dk
source,,,,,,,,,,
#Sverigeupproret#,0,0,0,0,0,0,0,0,0,0
"1,8 Miljoner Missnöjda",0,0,0,0,0,0,0,0,0,0
112Nyt,0,0,0,0,0,0,0,0,0,0
180Grader,1,0,0,0,0,0,0,0,0,0
24NYT,0,0,1,0,0,0,0,0,0,0
2860 Søborg uden filter.,0,0,0,0,0,0,0,0,0,0
4130 Viby Sjælland,0,0,0,0,0,0,0,0,0,0
4200 Alt er Tilladt,0,0,0,0,0,0,0,0,0,0
5G - Copenhagen Under Attack,0,0,0,0,0,0,0,0,0,0


## 6) Matrix Multiplication Intuition

**Goal:** From the actor–domain matrix **B**, we get an **actor–actor** matrix: “how many domains do actor i and actor j both link to?” That is **B @ B.T**: each entry (i, j) is the number of shared domains. This is the matrix way to **project** the bipartite graph onto actors (same idea as “two actors are tied if they share at least one domain”).

In [10]:
# 1) Actor–actor matrix = B @ B.T (shared-domain counts)
A_actor = B.values @ B.values.T
A_actor_df = pd.DataFrame(A_actor, index=B.index, columns=B.index)

print('Actor-actor shared-domain matrix shape:', A_actor_df.shape)
A_actor_df.iloc[:8, :8]

Actor-actor shared-domain matrix shape: (1077, 1077)


source,#Sverigeupproret#,"1,8 Miljoner Missnöjda",112Nyt,180Grader,24NYT,2860 Søborg uden filter.,4130 Viby Sjælland,4200 Alt er Tilladt
source,,,,,,,,
#Sverigeupproret#,1,0,0,0,0,0,0,0
"1,8 Miljoner Missnöjda",0,1,0,0,0,0,0,0
112Nyt,0,0,1,0,1,1,1,1
180Grader,0,0,0,1,0,0,0,0
24NYT,0,0,1,0,3,1,1,1
2860 Søborg uden filter.,0,0,1,0,1,1,1,1
4130 Viby Sjælland,0,0,1,0,1,1,2,1
4200 Alt er Tilladt,0,0,1,0,1,1,1,2


## 7) Undirected Actor Projection as Edge List

**Goal:** An edge list for the **actor–actor** network: one row per pair of actors that share at least one domain. **Weight** = number of domains they share. We do this by grouping the directed edges by domain, then for each domain with ≥2 actors we add an (undirected) edge between every pair; we aggregate so each (actor A, actor B) appears once with a total weight.

In [11]:
# 1) For each domain, get the list of actors that link to it
actor_domains = edge_dir[['source', 'target']].drop_duplicates()
shared = {}
for domain, g in actor_domains.groupby('target'):
    actors = sorted(g['source'].tolist())
    if len(actors) < 2:
        continue
    for a, b in combinations(actors, 2):
        shared.setdefault((a, b), set()).add(domain)

# 2) One row per (actor, actor) with weight = number of shared domains
rows = []
for (a, b), doms in shared.items():
    rows.append({'source': a, 'target': b, 'weight': len(doms), 'shared_domains': ';'.join(sorted(doms))})

edge_undir = pd.DataFrame(rows)

print('Undirected actor-actor edges:', len(edge_undir))
edge_undir.head()

Undirected actor-actor edges: 214694


,source,target,weight,shared_domains
0,180Grader,Atomkraft - ja tak!,1,180grader.dk
1,180Grader,Borgerlig debat,1,180grader.dk
2,180Grader,Claus Bøgh Svenningsen,1,180grader.dk
3,180Grader,Dane,1,180grader.dk
4,180Grader,Dansk Folkeparti Middelfart,1,180grader.dk


In [12]:
# Save the actor–actor edge list
edge_undir.to_csv('actor_actor_edge_list.csv', index=False)
print('Saved: actor_actor_edge_list.csv')

Saved: actor_actor_edge_list.csv


## 8) Build Actor Graph + Centrality

**Goal:** Build the undirected actor graph from the edge list, then compute **centrality** (degree, closeness, betweenness, eigenvector) and **communities**. We also attach each actor’s most frequent `account_type` so we can colour nodes. The table at the end summarises a few top actors by each measure.

In [13]:
# 1) Build undirected graph from actor–actor edges
Gu = nx.from_pandas_edgelist(edge_undir, 'source', 'target', edge_attr='weight', create_using=nx.Graph())

# 2) Centrality measures
degree = dict(Gu.degree())
degree_centrality = nx.degree_centrality(Gu)
closeness_centrality = nx.closeness_centrality(Gu)
betweenness_centrality = nx.betweenness_centrality(Gu, weight=None)
eigenvector_centrality = nx.eigenvector_centrality(Gu, max_iter=5000)

# 3) Communities (for grouping / colouring)
communities = list(greedy_modularity_communities(Gu))
community_map = {n: i for i, c in enumerate(communities) for n in c}

# 4) Most frequent account_type per actor (for labels)
acc = (work.groupby('actor_name', as_index=False)['account_type']
       .agg(lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else ''))
acc_map = dict(zip(acc['actor_name'], acc['account_type']))

# 5) Summary table: one row per actor with centralities and account_type
metrics = pd.DataFrame({
    'actor_name': list(Gu.nodes()),
    'account_type': [acc_map.get(n, '') for n in Gu.nodes()],
    'degree': [degree[n] for n in Gu.nodes()],
    'degree_centrality': [degree_centrality[n] for n in Gu.nodes()],
    'closeness_centrality': [closeness_centrality[n] for n in Gu.nodes()],
    'betweenness_centrality': [betweenness_centrality[n] for n in Gu.nodes()],
    'eigenvector_centrality': [eigenvector_centrality[n] for n in Gu.nodes()],
    'community': [community_map.get(n, -1) for n in Gu.nodes()]
}).sort_values(['degree','degree_centrality','betweenness_centrality'], ascending=False)

metrics.head(10)

,actor_name,account_type,degree,degree_centrality,closeness_centrality,betweenness_centrality,eigenvector_centrality,community
10,Klimarealisme.dk,facebook_group,908,0.854186,0.862061,0.066356,0.041730,1
1,Atomkraft - ja tak!,facebook_group,814,0.765757,0.796863,0.019207,0.041617,1
36,Irene,twitter_account,794,0.746943,0.785462,0.010376,0.041275,1
29,Det Nationale Danmarks Natur,facebook_group,793,0.746002,0.784281,0.005394,0.041593,1
43,PV- som statsminister,facebook_group,790,0.743180,0.782516,0.005196,0.041579,1
56,"Vi, der støtter Venstre.",facebook_group,781,0.734713,0.777268,0.004147,0.041572,1
22,Bent Preben Nielsen,twitter_account,774,0.728128,0.773809,0.006505,0.041188,1
75,Danexit,facebook_group,763,0.717780,0.766981,0.004544,0.041321,1
96,Jens Vornøe DF,facebook_page,762,0.716839,0.765855,0.002752,0.041499,1
106,Lis Jeppesen,twitter_account,761,0.715898,0.766417,0.007883,0.041140,1


** calculate the degree centrality and betweenness centrality of notes in a network**




In [14]:
metrics = pd.DataFrame({
    'actor_name': list(Gu.nodes()),
    'degree_centrality': [degree_centrality[n] for n in Gu.nodes()],
    'betweenness_centrality': [betweenness_centrality[n] for n in Gu.nodes()]
}).sort_values('degree_centrality', ascending=False)

print(metrics.head(10))

                       actor_name  degree_centrality  betweenness_centrality
10               Klimarealisme.dk           0.854186                0.066356
1             Atomkraft - ja tak!           0.765757                0.019207
36                          Irene           0.746943                0.010376
29   Det Nationale Danmarks Natur           0.746002                0.005394
43          PV- som statsminister           0.743180                0.005196
56       Vi, der støtter Venstre.           0.734713                0.004147
22            Bent Preben Nielsen           0.728128                0.006505
75                        Danexit           0.717780                0.004544
96                 Jens Vornøe DF           0.716839                0.002752
106                  Lis Jeppesen           0.715898                0.007883


In [ ]:
metrics.to_csv('actor_network_metrics.csv', index=False)
metrics.to_excel('actor_network_metrics.xlsx', index=False)
print('Saved: actor_network_metrics.csv')
print('Saved: actor_network_metrics.xlsx')

## 9) Add Node Attributes for Gephi

**Goal:** Attach the label columns (account_anno, actor_anno, emne_profil, platform, account_type) to each node so that in a visualization tool - e.g. Gephi - you can colour or filter by them. We take the most frequent value per actor from `work`, then write both graphs (directed actor–domain and undirected actor–actor) as `.gexf` files for Gephi.

In [ ]:
# 1) Most frequent attribute value per actor
attr_cols = ['account_anno', 'actor_anno', 'emne_profil', 'platform', 'account_type']
attrs = (work.groupby('actor_name', as_index=False)[attr_cols]
         .agg(lambda s: s.dropna().mode().iloc[0] if not s.dropna().empty else ''))
attr_map = attrs.set_index('actor_name').to_dict(orient='index')

# 2) Attach attributes to directed graph (actor and domain nodes)
for n in Gd.nodes():
    if n in attr_map:
        Gd.nodes[n].update(attr_map[n])
        Gd.nodes[n]['node_type'] = 'actor'
    else:
        Gd.nodes[n]['node_type'] = 'domain'

# 3) Attach attributes to undirected actor graph
for n in Gu.nodes():
    Gu.nodes[n].update(attr_map.get(n, {}))
    Gu.nodes[n]['node_type'] = 'actor'

# 4) Save as Gephi format
nx.write_gexf(Gd, 'actor_domain_directed.gexf')
nx.write_gexf(Gu, 'actor_actor_undirected.gexf')

print('Saved Gephi files:')
print('- datathings_teaching/student_actor_domain_directed.gexf')
print('- datathings_teaching/student_actor_actor_undirected.gexf')

## 10) Interpretation: Questions and Key Concepts

Answer the following in your own words. Use the outputs from the notebook above.

**Question(s):**

1. **Degree vs betweenness:** Which actors are high in degree centrality and which are high in betweenness? Why might the same actor score differently on the two measures?

2. **Projected graph:** The actor–actor graph is built from *shared domains*. How does this change the meaning of a "tie" compared with the original actor→domain graph? What are we measuring when we look at centrality in the projected graph?

3. **Edge list vs matrix:** What information is gained or lost when we convert the edge list to an adjacency (or incidence) matrix and back? When would you prefer one representation over the other?

4. **Weighted edges:** In this data, edge weight can reflect how often an actor links to a domain. How would using these weights (instead of binary edges) change your interpretation of centrality? Give one concrete example.